<a href="https://colab.research.google.com/github/IvanMorsin/Forecasting-electrical-power-in-multi-storey-residential-buildings/blob/main/notebook_9_XGBoost_LightGBM_CatBoost_RF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!pip install workalendar -q

In [11]:
!pip install kaleido==0.2.1 -q

In [12]:
!pip install catboost -q

In [13]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from sklearn.model_selection import TimeSeriesSplit
from workalendar.europe import Russia
from tqdm import tqdm
import os

In [14]:
def save_predictions(ts, house_ids, y_true, y_pred, horizon_name, model_name, filename):
    df_pred = pd.DataFrame({
        'timestamp': ts,
        'house_id': house_ids,
        'y_true': y_true,
        'y_pred': y_pred,
        'horizon': horizon_name,
        'model': model_name,
    })
    if os.path.exists(filename):
        df_existing = pd.read_csv(filename)
        df_pred = pd.concat([df_existing, df_pred], ignore_index=True)
    df_pred.to_csv(filename, index=False)

In [15]:
df = pd.read_csv('df_final+whether.csv', parse_dates=['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)

HOUSE_META = {
    'house_1': {'n_flats': 383, 'n_floors': 12},
    'house_2': {'n_flats': 191, 'n_floors': 12},
    'house_3': {'n_flats': 124, 'n_floors': 12},
    'house_4': {'n_flats': 263, 'n_floors': 12},
    'house_5': {'n_flats': 127, 'n_floors': 7},
    'house_6': {'n_flats': 497, 'n_floors': 25},
    'house_7': {'n_flats': 471, 'n_floors': 17},
    'house_8': {'n_flats': 171, 'n_floors': 23},
}
HOUSES = list(HOUSE_META.keys())

cal = Russia()
def is_holiday(dt):
    if dt.weekday() >= 5:
        return 0
    return int(not cal.is_working_day(dt.date()))

df['is_holiday'] = df['timestamp'].apply(is_holiday)

def evaluate(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    return {'MAE': round(mae, 3), 'RMSE': round(rmse, 3), 'MAPE': round(mape, 3)}

horizons = {
    '4h': 8,
    '8h': 16,
    '24h': 48,
    '7d': 336,
    '14d': 672,
    '1m': 1488,
}

# split по времени
split_train = df['timestamp'].quantile(0.70)
split_val = df['timestamp'].quantile(0.85)

# фолды по уникальным timestamps
unique_ts = df['timestamp'].values
tscv = TimeSeriesSplit(n_splits=5)
folds = []
for train_idx, val_idx in tscv.split(unique_ts):
    folds.append((unique_ts[train_idx[-1]], unique_ts[val_idx[-1]]))

for i, (t_tr, t_vl) in enumerate(folds):
    print(f'  Фолд {i+1}: train до {pd.Timestamp(t_tr).date()} | val до {pd.Timestamp(t_vl).date()}')

  Фолд 1: train до 2018-03-16 | val до 2018-07-20
  Фолд 2: train до 2018-07-20 | val до 2018-11-23
  Фолд 3: train до 2018-11-23 | val до 2019-03-29
  Фолд 4: train до 2019-03-29 | val до 2019-08-02
  Фолд 5: train до 2019-08-02 | val до 2019-12-06


In [16]:
scalers = {}

def make_features_all(df):
    frames = []
    for house, meta in HOUSE_META.items():
        data = df[["timestamp", house]].copy()
        data = data.rename(columns={house: "power"})

        train_mask = data["timestamp"] <= split_train
        scaler = MinMaxScaler()
        scaler.fit(data.loc[train_mask, ["power"]])
        scalers[house] = scaler
        data["power"] = scaler.transform(data[["power"]]).flatten()

        data["hour"] = data["timestamp"].dt.hour
        data["minute"] = data["timestamp"].dt.minute
        data["weekday"] = data["timestamp"].dt.weekday
        data["month"] = data["timestamp"].dt.month
        data["day_of_year"] = data["timestamp"].dt.dayofyear
        data["is_weekend"] = (data["weekday"] >= 5).astype(int)
        data["is_holiday"] = df["is_holiday"].values

        for lag in [1, 2, 48, 96, 336, 672, 1488]:
            data[f"lag_{lag}"] = data["power"].shift(lag)

        data["rolling_mean_48"] = data["power"].shift(1).rolling(48).mean()
        data["rolling_mean_336"] = data["power"].shift(1).rolling(336).mean()
        data["rolling_mean_672"] = data["power"].shift(1).rolling(672).mean()
        data["rolling_mean_1488"] = data["power"].shift(1).rolling(1488).mean()

        data["temp_c"] = df["temp_c"].values
        data["humidity"] = df["humidity"].values
        data["cloudiness"] = df["cloudiness"].values

        data["n_flats"] = meta["n_flats"]
        data["n_floors"] = meta["n_floors"]

        data["month_sin"] = np.sin(2 * np.pi * data["month"] / 12)
        data["month_cos"] = np.cos(2 * np.pi * data["month"] / 12)

        data["week_of_year"] = data["timestamp"].dt.isocalendar().week
        max_week = 52.0
        data["week_norm"] = (data["week_of_year"] - 1) / max_week
        data["week_sin"] = np.sin(2 * np.pi * data["week_norm"])
        data["week_cos"] = np.cos(2 * np.pi * data["week_norm"])

        data["house_id"] = house
        frames.append(data)

    df_all = pd.concat(frames, ignore_index=True)
    df_all = df_all.sort_values(["timestamp", "house_id"]).reset_index(drop=True)
    df_all = df_all.dropna().reset_index(drop=True)
    return df_all


def inverse_power(house, values):
    return scalers[house].inverse_transform(
        np.array(values).reshape(-1, 1)
    ).flatten()


df_long = make_features_all(df)

feature_cols = [
    c for c in df_long.columns
    if c not in ["timestamp", "power", "house_id"]
]

print(f"Строк: {len(df_long)}")
print(f"Признаки: {feature_cols}")

Строк: 278176
Признаки: ['hour', 'minute', 'weekday', 'month', 'day_of_year', 'is_weekend', 'is_holiday', 'lag_1', 'lag_2', 'lag_48', 'lag_96', 'lag_336', 'lag_672', 'lag_1488', 'rolling_mean_48', 'rolling_mean_336', 'rolling_mean_672', 'rolling_mean_1488', 'temp_c', 'humidity', 'cloudiness', 'n_flats', 'n_floors', 'month_sin', 'month_cos', 'week_of_year', 'week_norm', 'week_sin', 'week_cos']


In [17]:
os.makedirs("models", exist_ok=True)
xgb_rows = []

for horizon_name, horizon_points in tqdm(horizons.items(), desc="XGBoost"):

    frames_shifted = []
    for house in HOUSES:
        df_h = df_long[df_long["house_id"] == house].copy()
        df_h["power_target"] = df_h["power"].shift(-horizon_points)
        frames_shifted.append(df_h)

    df_shifted = pd.concat(frames_shifted, ignore_index=True)
    df_shifted = df_shifted.dropna(subset=["power_target"]).reset_index(drop=True)
    ts_shifted = df_shifted["timestamp"]

    fold_metrics_all = []

    for t_train_end, t_val_end in folds:
        mask_tr = ts_shifted <= t_train_end
        X_tr = df_shifted.loc[mask_tr, feature_cols]
        y_tr = df_shifted.loc[mask_tr, "power_target"]

        if X_tr.empty:
            continue

        model = XGBRegressor(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            n_jobs=-1,
            verbosity=0,
        )
        model.fit(X_tr, y_tr)

        for house in HOUSES:
            mask_h = (
                (ts_shifted > t_train_end)
                & (ts_shifted <= t_val_end)
                & (df_shifted["house_id"] == house)
            )
            X_vl_h = df_shifted.loc[mask_h, feature_cols].head(horizon_points)
            y_vl_h = df_shifted.loc[mask_h, "power_target"].head(horizon_points)
            ts_vl_h = df_shifted.loc[mask_h, "timestamp"].head(horizon_points)

            if len(y_vl_h) < horizon_points:
                continue

            y_pred_h = model.predict(X_vl_h)

            y_true_orig = inverse_power(house, y_vl_h.values)
            y_pred_orig = inverse_power(house, y_pred_h)

            fold_metrics_all.append({
                "house": house,
                **evaluate(y_true_orig, y_pred_orig),
            })

            if t_train_end == folds[-1][0]:
                save_predictions(
                    ts=ts_vl_h.values,
                    house_ids=np.full(len(y_vl_h), house),
                    y_true=y_true_orig,
                    y_pred=y_pred_orig,
                    horizon_name=horizon_name,
                    model_name="XGBoost",
                    filename="predictions_xgb.csv",
                )

    df_fold = pd.DataFrame(fold_metrics_all)
    for house in HOUSES:
        m = df_fold[df_fold["house"] == house]
        if len(m) == 0:
            continue
        xgb_rows.append({
            "модель": "XGBoost",
            "дом": house,
            "горизонт": horizon_name,
            "MAE": round(m["MAE"].mean(), 3),
            "RMSE": round(m["RMSE"].mean(), 3),
            "MAPE": round(m["MAPE"].mean(), 3),
        })

df_xgb = pd.DataFrame(xgb_rows)
df_xgb.to_csv("results_xgboost.csv", index=False)

print("XGBoost: MAPE:")
pivot = df_xgb.pivot(index="горизонт", columns="дом", values="MAPE")
pivot = pivot.reindex(list(horizons.keys()))
pivot["Среднее"] = pivot.mean(axis=1).round(2)
print(pivot.round(2).to_string())

XGBoost: 100%|██████████| 6/6 [06:28<00:00, 64.81s/it]

XGBoost: MAPE:
дом       house_1  house_2  house_3  house_4  house_5  house_6  house_7  house_8  Среднее
горизонт                                                                                 
4h           5.14     6.18     6.36     5.63     4.79     5.52     2.67     3.75     5.00
8h           6.06     7.19     7.89     5.10     6.18     4.94     4.74     4.92     5.88
24h          4.44     5.88     7.64     4.87     7.04     4.20     3.96     5.44     5.43
7d           5.32     6.52     7.48     5.11     6.68     6.23     4.23     5.23     5.85
14d          5.82     7.20     8.11     5.76     7.09     6.57     4.87     5.90     6.42
1m           8.29     9.19    11.11     9.23     9.17     8.46     6.76     7.87     8.76


In [18]:
import joblib
lgbm_rows = []
lgbm_final_models = {}

for horizon_name, horizon_points in tqdm(horizons.items(), desc="LightGBM"):

    frames_shifted = []
    for house in HOUSES:
        df_h = df_long[df_long["house_id"] == house].copy()
        df_h["power_target"] = df_h["power"].shift(-horizon_points)
        frames_shifted.append(df_h)

    df_shifted = pd.concat(frames_shifted, ignore_index=True)
    df_shifted = df_shifted.dropna(subset=["power_target"]).reset_index(drop=True)
    ts_shifted = df_shifted["timestamp"]

    fold_metrics_all = []

    for t_train_end, t_val_end in folds:
        mask_tr = ts_shifted <= t_train_end
        X_tr = df_shifted.loc[mask_tr, feature_cols]
        y_tr = df_shifted.loc[mask_tr, "power_target"]

        if X_tr.empty:
            continue

        model = LGBMRegressor(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            n_jobs=-1,
            verbose=-1,
        )
        model.fit(X_tr, y_tr)

        for house in HOUSES:
            mask_h = (
                (ts_shifted > t_train_end)
                & (ts_shifted <= t_val_end)
                & (df_shifted["house_id"] == house)
            )
            X_vl_h = df_shifted.loc[mask_h, feature_cols].head(horizon_points)
            y_vl_h = df_shifted.loc[mask_h, "power_target"].head(horizon_points)
            ts_vl_h = df_shifted.loc[mask_h, "timestamp"].head(horizon_points)

            if len(y_vl_h) < horizon_points:
                continue

            y_pred_h = model.predict(X_vl_h)

            y_true_orig = inverse_power(house, y_vl_h.values)
            y_pred_orig = inverse_power(house, y_pred_h)

            fold_metrics_all.append({
                "house": house,
                **evaluate(y_true_orig, y_pred_orig),
            })

            if t_train_end == folds[-1][0]:
                save_predictions(
                    ts=ts_vl_h.values,
                    house_ids=np.full(len(y_vl_h), house),
                    y_true=y_true_orig,
                    y_pred=y_pred_orig,
                    horizon_name=horizon_name,
                    model_name="LightGBM",
                    filename="predictions_lgbm.csv",
                )

    df_fold = pd.DataFrame(fold_metrics_all)
    for house in HOUSES:
        m = df_fold[df_fold["house"] == house]
        if len(m) == 0:
            continue
        lgbm_rows.append({
            "модель": "LightGBM",
            "дом": house,
            "горизонт": horizon_name,
            "MAE": round(m["MAE"].mean(), 3),
            "RMSE": round(m["RMSE"].mean(), 3),
            "MAPE": round(m["MAPE"].mean(), 3),
        })

    # финальная модель на полном train
    mask_full_tr = ts_shifted <= split_train
    X_full_tr = df_shifted.loc[mask_full_tr, feature_cols]
    y_full_tr = df_shifted.loc[mask_full_tr, "power_target"]

    final_model = LGBMRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    )
    final_model.fit(X_full_tr, y_full_tr)
    lgbm_final_models[horizon_name] = final_model
    joblib.dump(final_model, f"models/lgbm_{horizon_name}.pkl")
    print(f"  Сохранено: models/lgbm_{horizon_name}.pkl")

df_lgbm = pd.DataFrame(lgbm_rows)
df_lgbm.to_csv("results_lgbm.csv", index=False)

print("\nLightGBM: MAPE:")
pivot = df_lgbm.pivot(index="горизонт", columns="дом", values="MAPE")
pivot = pivot.reindex(list(horizons.keys()))
pivot["Среднее"] = pivot.mean(axis=1).round(2)
print(pivot.round(2).to_string())

LightGBM:  17%|█▋        | 1/6 [01:06<05:31, 66.38s/it]

  Сохранено: models/lgbm_4h.pkl


LightGBM:  33%|███▎      | 2/6 [02:17<04:36, 69.10s/it]

  Сохранено: models/lgbm_8h.pkl


LightGBM:  50%|█████     | 3/6 [03:23<03:22, 67.64s/it]

...

  Сохранено: models/lgbm_7d.pkl


LightGBM:  83%|████████▎ | 5/6 [05:28<01:04, 64.43s/it]

  Сохранено: models/lgbm_14d.pkl


LightGBM: 100%|██████████| 6/6 [06:38<00:00, 66.45s/it]

  Сохранено: models/lgbm_1m.pkl

LightGBM: MAPE:
дом       house_1  house_2  house_3  house_4  house_5  house_6  house_7  house_8  Среднее
горизонт                                                                                 
4h           5.39     5.75     7.68     5.65     4.38     5.70     2.69     3.69     5.12
8h           6.94     7.48     8.22     5.07     6.74     5.37     5.03     5.02     6.23
24h          4.75     5.95     7.73     4.89     7.02     4.09     3.79     5.47     5.46
7d           6.08     6.64     7.66     5.43     7.04     6.78     4.26     5.31     6.15
14d          5.93     7.26     8.18     5.76     7.32     6.49     4.88     5.85     6.46
1m           8.48     9.34    11.29     9.22     9.19     8.51     6.67     7.87     8.82


In [19]:
meta = {
    'houses':  HOUSES,
    'horizons':  horizons,
    'house_meta': HOUSE_META,
    'feature_cols': feature_cols,
    'split_train': str(split_train),
    'split_val': str(split_val),
}
joblib.dump(meta, 'models/model_meta.pkl')

['models/model_meta.pkl']

In [20]:
t_train_end, t_val_end = folds[-1]

fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[f"Горизонт {h}" for h in horizons.keys()],
    vertical_spacing=0.1,
    horizontal_spacing=0.08,
)

for i, (horizon_name, horizon_points) in enumerate(horizons.items()):
    row = i // 2 + 1
    col = i % 2 + 1

    frames_h = []
    for house in HOUSES:
        df_h = df_long[df_long["house_id"] == house].copy()
        df_h["power_target"] = df_h["power"].shift(-horizon_points)
        frames_h.append(df_h)
    df_h_all = pd.concat(frames_h, ignore_index=True)
    df_h_all = df_h_all.dropna(subset=["power_target"]).reset_index(drop=True)
    ts_h = df_h_all["timestamp"]

    mask_tr = ts_h <= t_train_end
    X_tr = df_h_all.loc[mask_tr, feature_cols]
    y_tr = df_h_all.loc[mask_tr, "power_target"]

    model_xgb_viz = XGBRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, n_jobs=-1, verbosity=0,
    )
    model_xgb_viz.fit(X_tr, y_tr)

    mask_vl = (
        (ts_h > t_train_end)
        & (ts_h <= t_val_end)
        & (df_h_all["house_id"] == "house_1")
    )
    X_vl = df_h_all.loc[mask_vl, feature_cols].head(horizon_points)
    y_vl = df_h_all.loc[mask_vl, "power_target"].head(horizon_points)
    ts_vl = df_h_all.loc[mask_vl, "timestamp"].head(horizon_points)

    if len(y_vl) < horizon_points:
        continue

    y_pred = model_xgb_viz.predict(X_vl)

    y_vl_orig = inverse_power("house_1", y_vl.values)
    y_pred_orig = inverse_power("house_1", y_pred)

    fig.add_trace(go.Scatter(
        x=ts_vl, y=y_vl_orig, mode="lines", name="факт",
        line=dict(color="#1f77b4", width=1.2, shape="hv"),
        showlegend=(i == 0),
    ), row=row, col=col)
    fig.add_trace(go.Scatter(
        x=ts_vl, y=y_pred_orig, mode="lines", name="прогноз XGBoost",
        line=dict(color="#d62728", width=1.2, shape="hv"),
        showlegend=(i == 0),
    ), row=row, col=col)

fig.update_layout(
    title="XGBoost: прогнозные и фактические значения по всем горизонтам (дом 1, fold 5)",
    width=700, height=900, font=dict(size=10),
    margin=dict(l=50, r=20, t=40, b=40), legend=dict(font=dict(size=9)),
)
fig.show()
fig.write_image("30_xgb_forecast_all_horizons.png", scale=2)

In [21]:
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[f"Горизонт {h}" for h in horizons.keys()],
    vertical_spacing=0.1,
    horizontal_spacing=0.08,
)

for i, (horizon_name, horizon_points) in enumerate(horizons.items()):
    row = i // 2 + 1
    col = i % 2 + 1

    frames_h = []
    for house in HOUSES:
        df_h = df_long[df_long["house_id"] == house].copy()
        df_h["power_target"] = df_h["power"].shift(-horizon_points)
        frames_h.append(df_h)
    df_h_all = pd.concat(frames_h, ignore_index=True)
    df_h_all = df_h_all.dropna(subset=["power_target"]).reset_index(drop=True)
    ts_h = df_h_all["timestamp"]

    mask_tr = ts_h <= t_train_end
    X_tr = df_h_all.loc[mask_tr, feature_cols]
    y_tr = df_h_all.loc[mask_tr, "power_target"]

    model_lgbm_viz = LGBMRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, n_jobs=-1, verbose=-1,
    )
    model_lgbm_viz.fit(X_tr, y_tr)

    mask_vl = (
        (ts_h > t_train_end)
        & (ts_h <= t_val_end)
        & (df_h_all["house_id"] == "house_1")
    )
    X_vl = df_h_all.loc[mask_vl, feature_cols].head(horizon_points)
    y_vl = df_h_all.loc[mask_vl, "power_target"].head(horizon_points)
    ts_vl = df_h_all.loc[mask_vl, "timestamp"].head(horizon_points)

    if len(y_vl) < horizon_points:
        continue

    y_pred = model_lgbm_viz.predict(X_vl)

    y_vl_orig = inverse_power("house_1", y_vl.values)
    y_pred_orig = inverse_power("house_1", y_pred)

    fig.add_trace(go.Scatter(
        x=ts_vl, y=y_vl_orig, mode="lines", name="факт",
        line=dict(color="#1f77b4", width=1.2, shape="hv"),
        showlegend=(i == 0),
    ), row=row, col=col)
    fig.add_trace(go.Scatter(
        x=ts_vl, y=y_pred_orig, mode="lines", name="прогноз LightGBM",
        line=dict(color="#d62728", width=1.2, shape="hv"),
        showlegend=(i == 0),
    ), row=row, col=col)

fig.update_layout(
    title="LightGBM: прогнозные и фактические значения по всем горизонтам (дом 1, fold 5)",
    width=700, height=900, font=dict(size=10),
    margin=dict(l=50, r=20, t=40, b=40), legend=dict(font=dict(size=9)),
)
fig.show()
fig.write_image("31_lgbm_forecast_all_horizons.png", scale=2)

In [22]:
# важность признаков, горизонт 24h
model_xgb_fi = model_xgb_viz   # последняя обученная XGBoost модель
model_lgbm_fi = lgbm_final_models['24h']

importances_xgb = model_xgb_fi.feature_importances_
importances_lgbm = model_lgbm_fi.feature_importances_

idx_xgb = np.argsort(importances_xgb)[::-1]
idx_lgbm = np.argsort(importances_lgbm)[::-1]

fig = make_subplots(rows=1, cols=2, subplot_titles=('XGBoost', 'LightGBM'))

fig.add_trace(go.Bar(
    x=[feature_cols[i] for i in idx_xgb],
    y=importances_xgb[idx_xgb],
    marker_color='#d62728', showlegend=False
), row=1, col=1)

fig.add_trace(go.Bar(
    x=[feature_cols[i] for i in idx_lgbm],
    y=importances_lgbm[idx_lgbm],
    marker_color='#1f77b4', showlegend=False
), row=1, col=2)

fig.update_layout(
    title='Важность признаков: горизонт 24h (общая модель)',
    width=900, height=400, font=dict(size=10),
    margin=dict(l=50, r=20, t=40, b=80)
)
fig.update_xaxes(tickangle=45)
fig.show()
fig.write_image('32_feature_importance.png', scale=2)

In [23]:
for i in idx_lgbm:
  print(feature_cols[i])

weekday
hour
day_of_year
lag_1
temp_c
rolling_mean_48
rolling_mean_336
rolling_mean_1488
rolling_mean_672
n_flats
lag_96
lag_336
humidity
lag_48
lag_2
week_cos
lag_1488
lag_672
week_sin
n_floors
week_of_year
cloudiness
month_cos
is_holiday
month
month_sin
minute
is_weekend
week_norm


In [24]:
for i in idx_xgb:
  print(feature_cols[i])

lag_96
lag_48
lag_1
lag_336
lag_1488
month_cos
lag_672
week_sin
hour
week_norm
weekday
is_weekend
lag_2
week_of_year
day_of_year
week_cos
month
month_sin
n_flats
rolling_mean_672
n_floors
rolling_mean_1488
rolling_mean_336
rolling_mean_48
temp_c
minute
is_holiday
humidity
cloudiness


In [25]:
#CatBoost
from catboost import CatBoostRegressor

catboost_rows = []

for horizon_name, horizon_points in tqdm(horizons.items(), desc="CatBoost"):

    frames_shifted = []
    for house in HOUSES:
        df_h = df_long[df_long["house_id"] == house].copy()
        df_h["power_target"] = df_h["power"].shift(-horizon_points)
        frames_shifted.append(df_h)

    df_shifted = pd.concat(frames_shifted, ignore_index=True)
    df_shifted = df_shifted.dropna(subset=["power_target"]).reset_index(drop=True)
    ts_shifted = df_shifted["timestamp"]

    fold_metrics_all = []

    for t_train_end, t_val_end in folds:
        mask_tr = ts_shifted <= t_train_end
        X_tr = df_shifted.loc[mask_tr, feature_cols]
        y_tr = df_shifted.loc[mask_tr, "power_target"]

        if X_tr.empty:
            continue

        model = CatBoostRegressor(
            iterations=500,
            learning_rate=0.05,
            depth=6,
            subsample=0.8,
            random_seed=42,
            verbose=0,
        )
        model.fit(X_tr, y_tr)

        for house in HOUSES:
            mask_h = (
                (ts_shifted > t_train_end)
                & (ts_shifted <= t_val_end)
                & (df_shifted["house_id"] == house)
            )
            X_vl_h = df_shifted.loc[mask_h, feature_cols].head(horizon_points)
            y_vl_h = df_shifted.loc[mask_h, "power_target"].head(horizon_points)
            ts_vl_h = df_shifted.loc[mask_h, "timestamp"].head(horizon_points)

            if len(y_vl_h) < horizon_points:
                continue

            y_pred_h = model.predict(X_vl_h)

            y_true_orig = inverse_power(house, y_vl_h.values)
            y_pred_orig = inverse_power(house, y_pred_h)

            fold_metrics_all.append({
                "house": house,
                **evaluate(y_true_orig, y_pred_orig),
            })

            if t_train_end == folds[-1][0]:
                save_predictions(
                    ts=ts_vl_h.values,
                    house_ids=np.full(len(y_vl_h), house),
                    y_true=y_true_orig,
                    y_pred=y_pred_orig,
                    horizon_name=horizon_name,
                    model_name="CatBoost",
                    filename="predictions_catboost.csv",
                )

    df_fold = pd.DataFrame(fold_metrics_all)
    for house in HOUSES:
        m = df_fold[df_fold["house"] == house]
        if len(m) == 0:
            continue
        catboost_rows.append({
            "модель": "CatBoost",
            "дом": house,
            "горизонт": horizon_name,
            "MAE": round(m["MAE"].mean(), 3),
            "RMSE": round(m["RMSE"].mean(), 3),
            "MAPE": round(m["MAPE"].mean(), 3),
        })

df_catboost = pd.DataFrame(catboost_rows)
df_catboost.to_csv("results_catboost.csv", index=False)

print("CatBoost: MAPE:")
pivot = df_catboost.pivot(index="горизонт", columns="дом", values="MAPE")
pivot = pivot.reindex(list(horizons.keys()))
pivot["Среднее"] = pivot.mean(axis=1).round(2)
print(pivot.round(2).to_string())

CatBoost: 100%|██████████| 6/6 [09:33<00:00, 95.55s/it]

CatBoost: MAPE:
дом       house_1  house_2  house_3  house_4  house_5  house_6  house_7  house_8  Среднее
горизонт                                                                                 
4h           7.07     5.55     7.63     4.25     5.42     6.36     3.18     3.53     5.37
8h           6.36     6.75     7.85     5.55     6.10     5.42     5.37     5.41     6.10
24h          4.83     6.06     7.49     5.14     6.99     4.06     4.18     5.63     5.55
7d           5.29     6.71     7.62     5.32     7.00     6.09     4.33     5.24     5.95
14d          5.44     7.31     8.00     5.91     6.91     6.24     5.19     5.80     6.35
1m           8.69     9.67    11.88     9.44     9.26     8.80     7.00     7.87     9.08


In [26]:
t_train_end, t_val_end = folds[-1]

fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[f'Горизонт {h}' for h in horizons.keys()],
    vertical_spacing=0.1, horizontal_spacing=0.08
)

for i, (horizon_name, horizon_points) in enumerate(horizons.items()):
    row = i // 2 + 1
    col = i % 2 + 1

    frames_h = []
    for house in HOUSES:
        df_h = df_long[df_long['house_id'] == house].copy()
        df_h['power_target'] = df_h['power'].shift(-horizon_points)
        frames_h.append(df_h)
    df_h_all = pd.concat(frames_h, ignore_index=True)
    df_h_all = df_h_all.dropna(subset=['power_target']).reset_index(drop=True)
    ts_h = df_h_all['timestamp']

    mask_tr = ts_h <= t_train_end
    X_tr = df_h_all.loc[mask_tr, feature_cols]
    y_tr = df_h_all.loc[mask_tr, 'power_target']

    model_viz = CatBoostRegressor(
        iterations=500, learning_rate=0.05, depth=6,
        subsample=0.8, random_seed=42, verbose=0
    )
    model_viz.fit(X_tr, y_tr)

    mask_vl = (ts_h > t_train_end) & \
              (ts_h <= t_val_end) & \
              (df_h_all['house_id'] == 'house_1')
    X_vl  = df_h_all.loc[mask_vl, feature_cols].head(horizon_points)
    y_vl  = df_h_all.loc[mask_vl, 'power_target'].head(horizon_points)
    ts_vl = df_h_all.loc[mask_vl, 'timestamp'].head(horizon_points)

    if len(y_vl) < horizon_points:
        continue

    y_pred = model_viz.predict(X_vl)

    fig.add_trace(go.Scatter(
        x=ts_vl, y=y_vl.values, mode='lines', name='факт',
        line=dict(color='#1f77b4', width=1.2, shape='hv'),
        showlegend=(i == 0)
    ), row=row, col=col)
    fig.add_trace(go.Scatter(
        x=ts_vl, y=y_pred, mode='lines', name='прогноз CatBoost',
        line=dict(color='#d62728', width=1.2, shape='hv'),
        showlegend=(i == 0)
    ), row=row, col=col)

fig.update_layout(
    title='CatBoost: прогнозные и фактические значения по всем горизонтам (дом 1, fold 5)',
    width=700, height=900, font=dict(size=10),
    margin=dict(l=50, r=20, t=40, b=40), legend=dict(font=dict(size=9))
)
fig.show()
fig.write_image('33_catboost_forecast_all_horizons.png', scale=2)

In [27]:
from sklearn.ensemble import RandomForestRegressor

rf_rows = []

for horizon_name, horizon_points in tqdm(horizons.items(), desc="RandomForest"):

    frames_shifted = []
    for house in HOUSES:
        df_h = df_long[df_long["house_id"] == house].copy()
        df_h["power_target"] = df_h["power"].shift(-horizon_points)
        frames_shifted.append(df_h)

    df_shifted = pd.concat(frames_shifted, ignore_index=True)
    df_shifted = df_shifted.dropna(subset=["power_target"]).reset_index(drop=True)
    ts_shifted = df_shifted["timestamp"]

    if len(df_shifted) == 0:
        print(f"Горизонт {horizon_name}: нет данных после сдвига")
        continue

    fold_metrics_all = []

    for t_train_end, t_val_end in folds:
        mask_tr = ts_shifted <= t_train_end
        if mask_tr.sum() < 100:
            continue

        X_tr = df_shifted.loc[mask_tr, feature_cols]
        y_tr = df_shifted.loc[mask_tr, "power_target"]

        sample_idx = X_tr.sample(frac=0.3, random_state=42).index
        X_tr_sample = X_tr.loc[sample_idx]
        y_tr_sample = y_tr.loc[sample_idx]

        model = RandomForestRegressor(
            n_estimators=100,
            max_depth=8,
            min_samples_leaf=10,
            random_state=42,
            n_jobs=-1,
        )
        model.fit(X_tr_sample, y_tr_sample)

        for house in HOUSES:
            mask_h = (
                (ts_shifted > t_train_end)
                & (ts_shifted <= t_val_end)
                & (df_shifted["house_id"] == house)
            )
            if mask_h.sum() < horizon_points:
                continue

            X_vl_h = df_shifted.loc[mask_h, feature_cols].head(horizon_points)
            y_vl_h = df_shifted.loc[mask_h, "power_target"].head(horizon_points)
            ts_vl_h = df_shifted.loc[mask_h, "timestamp"].head(horizon_points)

            y_pred_h = model.predict(X_vl_h)

            y_true_orig = inverse_power(house, y_vl_h.values)
            y_pred_orig = inverse_power(house, y_pred_h)

            fold_metrics_all.append({
                "house": house,
                **evaluate(y_true_orig, y_pred_orig),
            })

            if t_train_end == folds[-1][0]:
                save_predictions(
                    ts=ts_vl_h.values,
                    house_ids=np.full(len(y_vl_h), house),
                    y_true=y_true_orig,
                    y_pred=y_pred_orig,
                    horizon_name=horizon_name,
                    model_name="RandomForest",
                    filename="predictions_rf.csv",
                )

    if len(fold_metrics_all) == 0:
        print(f"Горизонт {horizon_name}: нет валидных предсказаний, пропускаем")
        continue

    df_fold = pd.DataFrame(fold_metrics_all)
    for house in HOUSES:
        m = df_fold[df_fold["house"] == house]
        if len(m) == 0:
            continue
        rf_rows.append({
            "модель": "RandomForest",
            "дом": house,
            "горизонт": horizon_name,
            "MAE": round(m["MAE"].mean(), 3),
            "RMSE": round(m["RMSE"].mean(), 3),
            "MAPE": round(m["MAPE"].mean(), 3),
        })

df_rf = pd.DataFrame(rf_rows)
df_rf.to_csv("results_rf.csv", index=False)

print("\nRandomForest: MAPE:")
pivot = df_rf.pivot(index="горизонт", columns="дом", values="MAPE")
pivot = pivot.reindex(list(horizons.keys()))
pivot["Среднее"] = pivot.mean(axis=1).round(2)
print(pivot.round(2).to_string())

RandomForest: 100%|██████████| 6/6 [24:20<00:00, 243.40s/it]


RandomForest: MAPE:
дом       house_1  house_2  house_3  house_4  house_5  house_6  house_7  house_8  Среднее
горизонт                                                                                 
4h           6.15     7.33     9.27     5.73     8.07     8.46     5.79     4.94     6.97
8h           6.48     7.79     9.42     6.93     7.55     6.10     5.84     6.05     7.02
24h          5.45     7.23     8.54     6.40     8.40     5.01     5.16     6.77     6.62
7d           5.49     6.97     7.98     5.89     7.57     6.72     4.72     5.50     6.36
14d          5.13     6.98     7.78     6.31     7.98     6.60     5.43     6.35     6.57
1m           9.98    10.55    12.48    11.86    11.01    10.23     8.13     9.13    10.42


In [28]:
df_pred_rf = pd.read_csv('predictions_rf.csv', parse_dates=['timestamp'])

fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[f'Горизонт {h}' for h in horizons.keys()],
    vertical_spacing=0.1, horizontal_spacing=0.08
)

for i, (horizon_name, horizon_points) in enumerate(horizons.items()):
    row = i // 2 + 1
    col = i % 2 + 1

    subset = df_pred_rf[
        (df_pred_rf['horizon'] == horizon_name) &
        (df_pred_rf['house_id'] == 'house_1')
    ].sort_values('timestamp').head(horizon_points)

    if len(subset) == 0:
        continue

    fig.add_trace(go.Scatter(
        x=subset['timestamp'], y=subset['y_true'],
        mode='lines', name='факт',
        line=dict(color='#1f77b4', width=1.2, shape='hv'),
        showlegend=(i == 0)
    ), row=row, col=col)

    fig.add_trace(go.Scatter(
        x=subset['timestamp'], y=subset['y_pred'],
        mode='lines', name='прогноз RandomForest',
        line=dict(color='#d62728', width=1.2, shape='hv'),
        showlegend=(i == 0)
    ), row=row, col=col)

    fig.update_xaxes(title_text='Дата', row=row, col=col, title_font=dict(size=9))
    fig.update_yaxes(title_text='кВт',  row=row, col=col, title_font=dict(size=9))

fig.update_layout(
    title='RandomForest: прогнозные и фактические значения по всем горизонтам (дом 1, fold 5)',
    width=700, height=900, font=dict(size=10),
    margin=dict(l=50, r=20, t=40, b=40), legend=dict(font=dict(size=9))
)
fig.show()
fig.write_image('34_rf_forecast_all_horizons.png', scale=2)

In [9]:
import joblib
import os

os.makedirs("models", exist_ok=True)

# сохраняем финальные модели LightGBM на полном train
for horizon_name, horizon_points in tqdm(horizons.items(), desc="Сохранение финальных моделей"):

    frames_shifted = []
    for house in HOUSES:
        df_h = df_long[df_long["house_id"] == house].copy()
        df_h["power_target"] = df_h["power"].shift(-horizon_points)
        frames_shifted.append(df_h)

    df_shifted = pd.concat(frames_shifted, ignore_index=True)
    df_shifted = df_shifted.dropna(subset=["power_target"]).reset_index(drop=True)
    ts_shifted = df_shifted["timestamp"]

    mask_full_tr = ts_shifted <= split_train
    X_full_tr = df_shifted.loc[mask_full_tr, feature_cols]
    y_full_tr = df_shifted.loc[mask_full_tr, "power_target"]

    final_model = LGBMRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    )
    final_model.fit(X_full_tr, y_full_tr)
    joblib.dump(final_model, f"models/lgbm_{horizon_name}.pkl")
    print(f"Сохранено: models/lgbm_{horizon_name}.pkl")

# сохраняем метаданные - скейлеры и feature_cols нужны для инференса
meta = {
    "houses": HOUSES,
    "horizons": horizons,
    "house_meta": HOUSE_META,
    "feature_cols": feature_cols,
    "split_train": str(split_train),
    "split_val": str(split_val),
    "scalers": scalers,  # MinMaxScaler per house
}
joblib.dump(meta, "models/model_meta.pkl")

Сохранение финальных моделей:  17%|█▋        | 1/6 [00:15<01:15, 15.09s/it]

Сохранено: models/lgbm_4h.pkl


Сохранение финальных моделей:  33%|███▎      | 2/6 [00:30<01:02, 15.52s/it]

Сохранено: models/lgbm_8h.pkl


Сохранение финальных моделей:  50%|█████     | 3/6 [00:46<00:46, 15.38s/it]

...

Сохранение финальных моделей:  83%|████████▎ | 5/6 [01:14<00:14, 14.76s/it]

Сохранено: models/lgbm_14d.pkl


Сохранение финальных моделей: 100%|██████████| 6/6 [01:32<00:00, 15.46s/it]

Сохранено: models/lgbm_1m.pkl


['models/model_meta.pkl']